<!-- ARTHASAATHI-PROVENANCE -->
> ## ⚠️ This notebook is demo evidence, not the running system
>
> The logic below was migrated to **`ml/src/councils/growth/credit_card.py`** and that module is what the application actually executes.
>
> **Role in the system:** Growth Council -> Credit Card Agent
>
> This notebook is preserved because its stored outputs are the record of the original analysis. **Editing the cells below will not change the system's behaviour** — change the module and its tests instead.
>
> Tests for this logic live under `tests/`, and every agent is covered by the cross-cutting contract suite in `tests/common/test_contracts.py`.


# 💳 Credit Card Recommendation Engine
**Two-part system:**
- **Part A** — Pure algorithmic scoring (no LLM)
- **Part B** — LLM-powered narrative final report via Anthropic API

> Built for Indian credit cards. Add more cards to `CARD_DATABASE` to expand coverage.


In [1]:

import json, os, math, textwrap, requests
from copy import deepcopy
from typing import Any

import os
import json
import pandas as pd

CARD_DATABASE= []

ROOT = r"final_decision"

for root, dirs, files in os.walk(ROOT):

    for file in files:

        if not file.endswith(".json"):
            continue

        path = os.path.join(root, file)

        try:
            with open(path, "r", encoding="utf-8") as f:
                card = json.load(f)

            CARD_DATABASE.append(card)

        except Exception as e:
            print(f"Failed: {path}")
            print(e)

print(f"Loaded {len(CARD_DATABASE)} cards")

Loaded 4 cards


## Step 1 — User Profile Input
Fill in the user's details and monthly spend breakdown.

In [3]:

# ════════════════════════════════════════════
#  USER PROFILE  — edit this section
# ════════════════════════════════════════════
USER_PROFILE = {
    # ── Personal info (for eligibility) ──
    "name":               "Rahul Sharma",
    "age":                28,
    "employment_type":    "salaried",   # "salaried" | "self_employed" | "student" | "unsalaried"
    "monthly_income":     80000,        # INR

    # ── Monthly spend (INR) per category ──
    "monthly_spend": {
        "online_shopping": 8000,
        "groceries":       5000,
        "dining":          4000,
        "fuel":            3000,
        "utility_bills":   3000,        # electricity, water, gas, telecom
        "travel":          5000,        # flights, hotels, cabs
        "entertainment":   2000,        # movies, OTT, events
        "international":   2000,        # forex spends
        "offline_retail":  4000,        # stores, general retail
        "others":          2000,
    },

    # ── Lifestyle flags ──
    "is_airtel_user":     True,         # uses Airtel telecom / Airtel Thanks App
    "is_amazon_prime":    True,
    "travel_frequency":   "occasional", # "none" | "occasional" | "frequent"
    "preferred_airlines": [],           # e.g. ["IndiGo", "Air India"]
    "preferred_hotels":   [],

    # ── Preferences ──
    "max_annual_fee":     2000,         # INR — max fee willing to pay
    "prefer_cashback":    True,
    "prefer_travel_perks":False,
}

total_monthly = sum(USER_PROFILE["monthly_spend"].values())
total_annual  = total_monthly * 12
print(f"👤 User: {USER_PROFILE['name']} | Age: {USER_PROFILE['age']} | Employment: {USER_PROFILE['employment_type']}")
print(f"💰 Total monthly spend: ₹{total_monthly:,}  |  Annual: ₹{total_annual:,}")
print()
print("📊 Monthly spend breakdown:")
for cat, amt in USER_PROFILE["monthly_spend"].items():
    pct = amt / total_monthly * 100
    bar = "█" * int(pct / 2)
    print(f"  {cat:<20} ₹{amt:>6,}  {bar} {pct:.1f}%")


👤 User: Rahul Sharma | Age: 28 | Employment: salaried
💰 Total monthly spend: ₹38,000  |  Annual: ₹456,000

📊 Monthly spend breakdown:
  online_shopping      ₹ 8,000  ██████████ 21.1%
  groceries            ₹ 5,000  ██████ 13.2%
  dining               ₹ 4,000  █████ 10.5%
  fuel                 ₹ 3,000  ███ 7.9%
  utility_bills        ₹ 3,000  ███ 7.9%
  travel               ₹ 5,000  ██████ 13.2%
  entertainment        ₹ 2,000  ██ 5.3%
  international        ₹ 2,000  ██ 5.3%
  offline_retail       ₹ 4,000  █████ 10.5%
  others               ₹ 2,000  ██ 5.3%


## Step 2 — Spend Category Analysis
Determine what type of card(s) the user needs based on spend ratios.

In [4]:

def analyze_spend_profile(profile: dict) -> dict:
    """
    Compute spend ratios and determine priority card categories.
    Returns a dict with ratios, dominant category, and recommended card types.
    """
    spend = profile["monthly_spend"]
    total = sum(spend.values()) or 1

    # Group into macro-categories
    macro = {
        "cashback_general": (
            spend.get("online_shopping", 0) +
            spend.get("offline_retail", 0) +
            spend.get("groceries", 0)
        ),
        "travel": (
            spend.get("travel", 0) +
            spend.get("international", 0)
        ),
        "lifestyle": (
            spend.get("dining", 0) +
            spend.get("entertainment", 0)
        ),
        "utility": spend.get("utility_bills", 0),
        "fuel":    spend.get("fuel", 0),
        "others":  spend.get("others", 0),
    }

    ratios = {k: v / total for k, v in macro.items()}

    # Travel gets a 1.4× alpha boost (per spec) because travel benefits
    # are typically high-value (lounge access, miles, hotel status)
    TRAVEL_ALPHA = 1.4
    weighted = {
        "CASHBACK": ratios["cashback_general"] + ratios["utility"] * 0.6,
        "TRAVEL":   (ratios["travel"] + ratios["fuel"] * 0.3) * TRAVEL_ALPHA,
        "LIFESTYLE":ratios["lifestyle"] + ratios["utility"] * 0.3,
        "REWARDS":  ratios["others"] * 0.5,
    }

    # Sort by weighted importance
    ranked = sorted(weighted.items(), key=lambda x: -x[1])
    dominant = ranked[0][0]

    # User adjustments
    if profile.get("prefer_travel_perks") and profile.get("travel_frequency") == "frequent":
        dominant = "TRAVEL"
    if profile.get("prefer_cashback"):
        # nudge cashback up if user prefers it
        weighted["CASHBACK"] *= 1.15

    return {
        "macro_spend":   macro,
        "ratios":        ratios,
        "weighted":      weighted,
        "ranked_types":  ranked,
        "dominant_type": dominant,
        "total_monthly": total,
        "total_annual":  total * 12,
    }


spend_analysis = analyze_spend_profile(USER_PROFILE)

print("═" * 55)
print("  SPEND CATEGORY ANALYSIS")
print("═" * 55)
print(f"  Dominant card type needed: ► {spend_analysis['dominant_type']}")
print()
print("  Weighted category scores (higher = more important):")
for cat, score in sorted(spend_analysis["weighted"].items(), key=lambda x: -x[1]):
    bar = "▓" * int(score * 40)
    print(f"  {cat:<12} {bar} {score:.3f}")
print()
print("  Macro spend buckets (monthly):")
for k, v in spend_analysis["macro_spend"].items():
    print(f"  {k:<22} ₹{v:>6,}")
print("═" * 55)


═══════════════════════════════════════════════════════
  SPEND CATEGORY ANALYSIS
═══════════════════════════════════════════════════════
  Dominant card type needed: ► CASHBACK

  Weighted category scores (higher = more important):
  CASHBACK     ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓ 0.569
  TRAVEL       ▓▓▓▓▓▓▓▓▓▓▓ 0.291
  LIFESTYLE    ▓▓▓▓▓▓▓ 0.182
  REWARDS      ▓ 0.026

  Macro spend buckets (monthly):
  cashback_general       ₹17,000
  travel                 ₹ 7,000
  lifestyle              ₹ 6,000
  utility                ₹ 3,000
  fuel                   ₹ 3,000
  others                 ₹ 2,000
═══════════════════════════════════════════════════════


## Step 3 — Eligibility Filter
Remove cards the user cannot apply for.

In [5]:

def filter_eligible_cards(profile: dict, cards: list) -> tuple[list, list]:
    """
    Returns (eligible_cards, rejected_cards_with_reason).
    """
    eligible, rejected = [], []

    age  = profile["age"]
    emp  = profile["employment_type"]          # salaried | self_employed | student | unsalaried
    fee  = profile.get("max_annual_fee", 99999)

    for card in cards:
        reasons = []

        # Age check
        if card.get("age_min") is None or card.get("age_max") is None:
           continue  # skip cards with missing age criteria
        elif age < card.get("age_min", 0):
            reasons.append(f"Age {age} below minimum {card['age_min']}")
        elif age > card.get("age_max", 100):
            reasons.append(f"Age {age} above maximum {card['age_max']}")

        # Employment check
        if emp == "student"       and not card.get("student_eligible",      False):
            reasons.append("Card not available for students")
        if emp == "self_employed" and not card.get("self_employed_eligible", True):
            reasons.append("Card not available for self-employed")
        if emp == "unsalaried"    and not card.get("unsalaried_eligible",    False):
            reasons.append("Card not available for unsalaried individuals")

        # Invite-only
        if card.get("invite_only", False):
            reasons.append("Invite-only card")

        # Annual fee preference
        eff_fee = card.get("annual_fee", 0)
        if eff_fee > fee:
            reasons.append(f"Annual fee ₹{eff_fee} exceeds user max ₹{fee}")

        if reasons:
            rejected.append({"card": card["card_name"], "reasons": reasons})
        else:
            eligible.append(card)

    return eligible, rejected


eligible_cards, rejected_cards = filter_eligible_cards(USER_PROFILE, CARD_DATABASE)

print(f"✅ Eligible cards  : {len(eligible_cards)}")
for c in eligible_cards:
    print(f"   • {c['card_name']}  (fee: ₹{c['annual_fee']})")

print(f"\n❌ Rejected cards  : {len(rejected_cards)}")
for r in rejected_cards:
    print(f"   • {r['card']}  → {'; '.join(r['reasons'])}")


✅ Eligible cards  : 3
   • Airtel Axis Bank Credit Card  (fee: ₹500)
   • Axis Bank Ace Credit Card  (fee: ₹499)
   • Axis Bank Aura Credit Card  (fee: ₹749)

❌ Rejected cards  : 1
   • Axis Bank Atlas Credit Card  → Annual fee ₹5000 exceeds user max ₹2000


## Step 4 — Per-Card Benefit Calculator
Map user's actual spend to each card's reward rates and compute annual savings.

In [6]:

def calculate_card_value(card: dict, profile: dict, spend_analysis: dict) -> dict:
    """
    Estimate annual monetary value a user gets from a card.
    Returns a breakdown dict with per-category savings and a total.
    """
    spend  = profile["monthly_spend"]
    is_airtel = profile.get("is_airtel_user", False)
    is_prime  = profile.get("is_amazon_prime", False)

    breakdown = {}

    # ── 1. Base cashback / rewards on all spend ──────────────────────
    base_rate = card.get("base_reward_rate", 1) / 100   # to decimal
    all_spend_annual = spend_analysis["total_annual"]
    breakdown["base_rewards"] = all_spend_annual * base_rate

    # ── 2. Category-specific rates ──────────────────────────────────
    # Utility
    util_rate = card.get("utility_reward_rate")
    if util_rate:
        util_annual = spend.get("utility_bills", 0) * 12
        if not is_airtel and "Airtel" in card["card_name"]:
            util_rate = 2   # non-Airtel users get reduced rate on Airtel card
        extra = util_annual * (util_rate - card.get("base_reward_rate", 1)) / 100
        breakdown["utility_bonus"] = max(extra, 0)

    # Travel
    travel_rate = card.get("travel_reward_rate")
    if travel_rate:
        travel_annual = spend.get("travel", 0) * 12
        extra = travel_annual * (travel_rate - card.get("base_reward_rate", 1)) / 100
        breakdown["travel_bonus"] = max(extra, 0)

    # ── 3. Named benefit valuation ───────────────────────────────────
    named_value = 0
    named_detail = []

    for b in card.get("benefits", []):
        bcat  = b.get("benefit_category", "")
        bval  = b.get("value", 0)
        bunit = b.get("value_unit", "")
        blim  = b.get("max_limit")
        bper  = b.get("limit_period", "")
        periods_per_year = 12 if "Monthly" in (bper or "") else 1

        # Skip non-percentage benefits from value calc (eVouchers handled separately)
        if bunit == "INR":
            named_value += bval        # one-time welcome vouchers etc.
            named_detail.append(f"{b['company_name']}: ₹{bval} voucher")
            continue

        if bunit != "%":
            continue

        # Map benefit to user spend
        company = b.get("company_name", "").lower()
        user_monthly_spend = 0

        if "zomato" in company or "swiggy" in company:
            user_monthly_spend = spend.get("dining", 0) * 0.5
        elif "blinkit" in company:
            user_monthly_spend = spend.get("groceries", 0) * 0.4
        elif "airtel" in company:
            user_monthly_spend = spend.get("utility_bills", 0) * (0.8 if is_airtel else 0)
        elif "utility" in company:
            user_monthly_spend = spend.get("utility_bills", 0) * (0.7 if is_airtel else 0)
        elif "amazon" in company:
            user_monthly_spend = spend.get("online_shopping", 0) * 0.5
        elif "goibibo" in company or "makemytrip" in company:
            user_monthly_spend = spend.get("travel", 0) * 0.4
        elif "fuel" in company:
            user_monthly_spend = spend.get("fuel", 0)
        elif "movie" in company or "district" in company:
            user_monthly_spend = spend.get("entertainment", 0) * 0.4
        elif "tira" in company:
            user_monthly_spend = spend.get("offline_retail", 0) * 0.1
        elif "online" in company:
            user_monthly_spend = spend.get("online_shopping", 0)
        elif "offline" in company:
            user_monthly_spend = spend.get("offline_retail", 0)
        elif "dining" in company:
            user_monthly_spend = spend.get("dining", 0)
        elif "travel" in company:
            user_monthly_spend = spend.get("travel", 0)

        if user_monthly_spend == 0:
            continue

        raw_monthly = user_monthly_spend * bval / 100
        if blim is not None:
            raw_monthly = min(raw_monthly, blim)

        annual = raw_monthly * periods_per_year
        named_value += annual
        named_detail.append(f"{b['company_name']} {bval}%: ₹{annual:,.0f}/yr")

    breakdown["named_benefits"] = named_value
    breakdown["named_detail"]   = named_detail

    # ── 4. Lounge access value ────────────────────────────────────────

    dom_lounges = card.get("domestic_lounge_visits") or 0
    intl_lounges = card.get("international_lounge_visits") or 0
    lounge_value = dom_lounges * 500 + intl_lounges * 2000   # approx market value
    breakdown["lounge_value"] = lounge_value

    # ── 5. Welcome bonus (one-time, amortise over 1 year) ────────────
    breakdown["welcome_bonus"] = card.get("welcome_bonus_value", 0)

    # ── 6. Forex savings ─────────────────────────────────────────────
    intl_spend_annual = spend.get("international", 0) * 12
    # Compare to avg 3.5% markup; card's markup is forex_markup
    avg_markup = 3.5
    card_markup = card.get("forex_markup", 3.5)
    forex_saving = intl_spend_annual * (avg_markup - card_markup) / 100
    breakdown["forex_savings"] = max(forex_saving, 0)

    # ── 7. Fee waiver check ──────────────────────────────────────────
    annual_fee = card.get("annual_fee", 0)
    waiver_threshold = card.get("fee_waiver_spend", None)
    if waiver_threshold and spend_analysis["total_annual"] >= waiver_threshold:
        breakdown["fee_waived"] = True
        effective_fee = 0
    else:
        breakdown["fee_waived"] = False
        effective_fee = annual_fee

    # ── Total ────────────────────────────────────────────────────────
    gross = (
        breakdown.get("base_rewards") or 0 +
        breakdown.get("utility_bonus") or 0 +
        breakdown.get("travel_bonus") or 0 +
        breakdown.get("named_benefits") or 0 +
        breakdown.get("lounge_value") or 0 +
        breakdown.get("welcome_bonus") or 0 +
        breakdown.get("forex_savings") or 0
    )

    # Dedup with base (named benefits may overlap)
    gross = gross * 0.75   # conservative overlap discount

    net = gross - effective_fee
    breakdown["gross_value"]   = gross
    breakdown["annual_fee"]    = annual_fee
    breakdown["effective_fee"] = effective_fee
    breakdown["net_value"]     = net

    return breakdown


# Score all eligible cards
card_scores = []
for card in eligible_cards:
    val = calculate_card_value(card, USER_PROFILE, spend_analysis)
    card_scores.append({
        "card":      card,
        "valuation": val,
        "net_value": val["net_value"],
    })

card_scores.sort(key=lambda x: -x["net_value"])

print("╔══════════════════════════════════════════════════════════╗")
print("║           CARD VALUATION RESULTS (Net Annual ₹)         ║")
print("╠══════════════════════════════════════════════════════════╣")
for i, cs in enumerate(card_scores, 1):
    card = cs["card"]
    v    = cs["valuation"]
    star = " ⭐" if i == 1 else ""
    print(f"║ #{i}  {card['card_name']:<38}{star}")
    print(f"║     Gross: ₹{v['gross_value']:>8,.0f}  Fee: ₹{v['effective_fee']:>5,}  Net: ₹{v['net_value']:>8,.0f}")
    if v.get("fee_waived"):
        print(f"║     ✓ Annual fee waived (spend threshold met)")
    print("║")
print("╚══════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════╗
║           CARD VALUATION RESULTS (Net Annual ₹)         ║
╠══════════════════════════════════════════════════════════╣
║ #1  Axis Bank Aura Credit Card             ⭐
║     Gross: ₹   6,840  Fee: ₹  749  Net: ₹   6,091
║
║ #2  Axis Bank Ace Credit Card             
║     Gross: ₹   5,130  Fee: ₹    0  Net: ₹   5,130
║     ✓ Annual fee waived (spend threshold met)
║
║ #3  Airtel Axis Bank Credit Card          
║     Gross: ₹   3,420  Fee: ₹    0  Net: ₹   3,420
║     ✓ Annual fee waived (spend threshold met)
║
╚══════════════════════════════════════════════════════════╝


## Step 5 — Category-Level Card Routing
Which card to use for which type of spend?

In [7]:

def build_spend_routing(card_scores: list, profile: dict) -> dict:
    """
    For each spend category, recommend the best eligible card.
    Applies travel alpha boost (1.4×) when comparing cards for travel spends.
    """
    TRAVEL_ALPHA = 1.4

    spend_categories = [
        "online_shopping", "groceries", "dining", "fuel",
        "utility_bills", "travel", "entertainment", "international",
        "offline_retail", "others",
    ]

    # Build a quick lookup: card_name -> benefit rates
    routing = {}

    for cat in spend_categories:
        best_card  = None
        best_score = -999
        best_reason = ""

        for cs in card_scores:
            card = cs["card"]
            val  = cs["valuation"]
            score = 0
            reason_parts = []

            # Category-specific scoring
            if cat == "utility_bills":
                rate = card.get("utility_reward_rate", card.get("base_reward_rate", 1))
                score = rate
                reason_parts.append(f"{rate}% cashback")

            elif cat == "travel":
                rate = card.get("travel_reward_rate", card.get("base_reward_rate", 1))
                lounge_bonus = (card.get("domestic_lounge_visits", 0) * 500 +
                                card.get("international_lounge_visits", 0) * 2000) / 12
                score = (rate * TRAVEL_ALPHA) + lounge_bonus / 1000
                reason_parts.append(f"{rate}% rewards × {TRAVEL_ALPHA} alpha")
                if lounge_bonus > 0:
                    reason_parts.append(f"lounge access")

            elif cat == "international":
                rate = card.get("international_reward_rate", card.get("base_reward_rate", 1))
                forex_save = max(0, 3.5 - card.get("forex_markup", 3.5))
                score = rate + forex_save
                reason_parts.append(f"{rate}% rewards + {forex_save:.1f}% forex saving")

            elif cat == "fuel":
                rate = card.get("fuel_reward_rate", card.get("base_reward_rate", 1))
                score = rate
                reason_parts.append(f"{rate}% / fuel surcharge waiver")

            elif cat == "dining":
                rate = card.get("dining_reward_rate", card.get("base_reward_rate", 1))
                # Check named benefits for dining
                for b in card.get("benefits", []):
                    if b.get("value_unit") == "%" and ("zomato" in b.get("company_name","").lower() or "swiggy" in b.get("company_name","").lower()):
                        rate = max(rate, b.get("value", 0))
                score = rate
                reason_parts.append(f"up to {rate}% on dining/food apps")

            elif cat in ("online_shopping", "groceries", "entertainment", "offline_retail", "others"):
                rate = card.get("base_reward_rate", 1)
                # Check named online benefits
                for b in card.get("benefits", []):
                    bname = b.get("company_name","").lower()
                    if ("online" in bname or "amazon" in bname or "flipkart" in bname):
                        rate = max(rate, b.get("value", 0))
                score = rate
                reason_parts.append(f"{rate}% cashback/rewards")

            if score > best_score:
                best_score  = score
                best_card   = card["card_name"]
                best_reason = ", ".join(reason_parts) if reason_parts else f"{score:.1f}%"

        routing[cat] = {
            "card":   best_card,
            "score":  best_score,
            "reason": best_reason,
        }

    return routing


routing = build_spend_routing(card_scores, USER_PROFILE)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║              USE THIS CARD FOR EACH CATEGORY                    ║")
print("╠══════════════════════════════════════════════════════════════════╣")
for cat, rec in routing.items():
    monthly_spend = USER_PROFILE["monthly_spend"].get(cat, 0)
    saving_est    = monthly_spend * rec["score"] / 100
    print(f"║  {cat:<22} → {rec['card']:<28}")
    print(f"║  {'':22}   Reason: {rec['reason']:<35}")
    print(f"║  {'':22}   Est. saving: ₹{saving_est:,.0f}/mo  |  ₹{saving_est*12:,.0f}/yr")
    print("║")
print("╚══════════════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════════════╗
║              USE THIS CARD FOR EACH CATEGORY                    ║
╠══════════════════════════════════════════════════════════════════╣
║  online_shopping        → Axis Bank Aura Credit Card  
║                           Reason: 750% cashback/rewards              
║                           Est. saving: ₹60,000/mo  |  ₹720,000/yr
║
║  groceries              → Axis Bank Aura Credit Card  
║                           Reason: 750% cashback/rewards              
║                           Est. saving: ₹37,500/mo  |  ₹450,000/yr
║
║  dining                 → Airtel Axis Bank Credit Card
║                           Reason: up to 10% on dining/food apps      
║                           Est. saving: ₹400/mo  |  ₹4,800/yr
║
║  fuel                   → Axis Bank Aura Credit Card  
║                           Reason: 0% / fuel surcharge waiver         
║                           Est. saving: ₹0/mo  |  ₹0/yr
║
║  utility_bil

## Part A — Final Recommendation (Algorithmic)
Top card picks with detailed monetary savings and benefits listing.

In [8]:

def print_algorithmic_recommendation(card_scores, routing, spend_analysis, profile):
    """Clean console report — no LLM required."""
    top_n = min(3, len(card_scores))
    print("\n" + "═"*65)
    print("  PART A: ALGORITHMIC CREDIT CARD RECOMMENDATION")
    print("═"*65)
    print(f"  User : {profile['name']}   |   Monthly Spend: ₹{spend_analysis['total_monthly']:,}")
    print(f"  Dominant Need: {spend_analysis['dominant_type']}")
    print("═"*65)

    for rank, cs in enumerate(card_scores[:top_n], 1):
        card = cs["card"]
        v    = cs["valuation"]
        print(f"\n{'▶' if rank==1 else ' '} #{rank}  {card['card_name']}  ({card['card_type']})")
        print(f"     Issuer: {card['issuer']}  |  Annual Fee: ₹{card['annual_fee']}", end="")
        if v.get("fee_waived"):
            print("  → WAIVED ✓", end="")
        print()
        print(f"     Best For: {', '.join(card.get('best_for', []))}")
        print(f"\n     💰 Annual Savings Breakdown:")
        print(f"        Base Rewards       ₹{v.get('base_rewards',0):>8,.0f}")
        if v.get("utility_bonus",   0) > 0: print(f"        Utility Bonus      ₹{v['utility_bonus']:>8,.0f}")
        if v.get("travel_bonus",    0) > 0: print(f"        Travel Bonus       ₹{v['travel_bonus']:>8,.0f}")
        if v.get("named_benefits",  0) > 0: print(f"        Partner Benefits   ₹{v['named_benefits']:>8,.0f}")
        if v.get("lounge_value",    0) > 0: print(f"        Lounge Access      ₹{v['lounge_value']:>8,.0f}")
        if v.get("forex_savings",   0) > 0: print(f"        Forex Savings      ₹{v['forex_savings']:>8,.0f}")
        if v.get("welcome_bonus",   0) > 0: print(f"        Welcome Bonus      ₹{v['welcome_bonus']:>8,.0f}")
        print(f"        {'─'*28}")
        print(f"        GROSS VALUE        ₹{v['gross_value']:>8,.0f}")
        print(f"        Less: Annual Fee  -₹{v['effective_fee']:>8,.0f}")
        print(f"        NET VALUE          ₹{v['net_value']:>8,.0f}  ← you save this")

        print(f"\n     📋 Key Benefits & Terms:")
        for b in card.get("benefits", []):
            cond = f" ({b['conditions']})" if b.get("conditions") else ""
            lim  = f" [max ₹{b['max_limit']}/{b.get('limit_period','mo')}]" if b.get("max_limit") else ""
            print(f"        • {b['company_name']}: {b['value']}{b['value_unit']} {b['benefit_category']}{lim}{cond}")

        excl = card.get("excluded_categories", [])
        if excl:
            print(f"\n     ⚠️  Excluded (no rewards): {', '.join(excl[:4])}{' ...' if len(excl)>4 else ''}")

    print("\n" + "─"*65)
    print("  📌 CARD USAGE ROUTING SUMMARY")
    print("─"*65)
    for cat, rec in routing.items():
        ms = profile["monthly_spend"].get(cat, 0)
        if ms == 0:
            continue
        print(f"  {cat:<22} → {rec['card']}")
    print("═"*65)


print_algorithmic_recommendation(card_scores, routing, spend_analysis, USER_PROFILE)



═════════════════════════════════════════════════════════════════
  PART A: ALGORITHMIC CREDIT CARD RECOMMENDATION
═════════════════════════════════════════════════════════════════
  User : Rahul Sharma   |   Monthly Spend: ₹38,000
  Dominant Need: CASHBACK
═════════════════════════════════════════════════════════════════

▶ #1  Axis Bank Aura Credit Card  (REWARDS)
     Issuer: Axis Bank  |  Annual Fee: ₹749
     Best For: Frequent travelers, Dining enthusiasts, Online shoppers, Health-conscious individuals, Fitness enthusiasts

     💰 Annual Savings Breakdown:
        Base Rewards       ₹   9,120
        Partner Benefits   ₹   2,000
        Forex Savings      ₹     480
        ────────────────────────────
        GROSS VALUE        ₹   6,840
        Less: Annual Fee  -₹     749
        NET VALUE          ₹   6,091  ← you save this

     📋 Key Benefits & Terms:
        • Decathlon: 750INR Shopping (Valid for 60 days from issuance. Limited to one-time usage. Non-refundable. Standard s

## Part B — LLM-Powered Final Report
### Step 6 — Prepare Context for Claude API

In [9]:

def build_llm_context(card_scores, routing, spend_analysis, profile, top_n=3) -> str:
    """Serialize the algorithmic outputs into a compact JSON payload for the LLM."""

    top_cards = []
    for cs in card_scores[:top_n]:
        card = cs["card"]
        v    = cs["valuation"]
        top_cards.append({
            "rank":          card_scores.index(cs) + 1,
            "card_name":     card["card_name"],
            "issuer":        card["issuer"],
            "card_type":     card["card_type"],
            "annual_fee":    card["annual_fee"],
            "fee_waived":    v.get("fee_waived", False),
            "effective_fee": v["effective_fee"],
            "gross_annual_value": round(v["gross_value"]),
            "net_annual_value":   round(v["net_value"]),
            "savings_breakdown": {
                "base_rewards":    round(v.get("base_rewards", 0)),
                "utility_bonus":   round(v.get("utility_bonus", 0)),
                "travel_bonus":    round(v.get("travel_bonus", 0)),
                "partner_benefits":round(v.get("named_benefits", 0)),
                "lounge_value":    round(v.get("lounge_value", 0)),
                "forex_savings":   round(v.get("forex_savings", 0)),
                "welcome_bonus":   round(v.get("welcome_bonus", 0)),
            },
            "partner_benefit_detail": v.get("named_detail", [])[:8],
            "best_for":    card.get("best_for", []),
            "avoid_for":   card.get("excluded_categories", [])[:5],
            "key_benefits": [
                {
                    "partner":    b["company_name"],
                    "rate":       f"{b['value']}{b['value_unit']}",
                    "type":       b["benefit_category"],
                    "cap":        f"max ₹{b['max_limit']}" if b.get("max_limit") else "uncapped",
                    "conditions": b.get("conditions", ""),
                }
                for b in card.get("benefits", [])
            ],
        })

    payload = {
        "user": {
            "name":             profile["name"],
            "age":              profile["age"],
            "employment":       profile["employment_type"],
            "monthly_spend":    profile["monthly_spend"],
            "total_monthly":    spend_analysis["total_monthly"],
            "total_annual":     spend_analysis["total_annual"],
            "is_airtel_user":   profile.get("is_airtel_user", False),
            "is_amazon_prime":  profile.get("is_amazon_prime", False),
            "travel_frequency": profile.get("travel_frequency", "occasional"),
            "max_annual_fee":   profile.get("max_annual_fee", 9999),
        },
        "spend_analysis": {
            "dominant_card_type": spend_analysis["dominant_type"],
            "ranked_types": [
                {"type": t, "weighted_score": round(s, 3)}
                for t, s in spend_analysis["ranked_types"]
            ],
        },
        "top_recommended_cards": top_cards,
        "spend_routing": {
            cat: {"card": r["card"], "reason": r["reason"]}
            for cat, r in routing.items()
            if profile["monthly_spend"].get(cat, 0) > 0
        },
    }

    return json.dumps(payload, indent=2)


llm_context = build_llm_context(card_scores, routing, spend_analysis, USER_PROFILE)
print("✅ LLM context prepared. Preview (first 800 chars):")
print(llm_context[:800], "...")
print(f"\nTotal context length: {len(llm_context)} characters")


✅ LLM context prepared. Preview (first 800 chars):
{
  "user": {
    "name": "Rahul Sharma",
    "age": 28,
    "employment": "salaried",
    "monthly_spend": {
      "online_shopping": 8000,
      "groceries": 5000,
      "dining": 4000,
      "fuel": 3000,
      "utility_bills": 3000,
      "travel": 5000,
      "entertainment": 2000,
      "international": 2000,
      "offline_retail": 4000,
      "others": 2000
    },
    "total_monthly": 38000,
    "total_annual": 456000,
    "is_airtel_user": true,
    "is_amazon_prime": true,
    "travel_frequency": "occasional",
    "max_annual_fee": 2000
  },
  "spend_analysis": {
    "dominant_card_type": "CASHBACK",
    "ranked_types": [
      {
        "type": "CASHBACK",
        "weighted_score": 0.495
      },
      {
        "type": "TRAVEL",
        "weighted_score": 0.291
      },
      {
 ...

Total context length: 9179 characters


### Step 7 — Generate Final Report via Claude API

In [12]:
from dotenv import load_dotenv
from groq import Groq



load_dotenv()  # Load environment variables from .env file (for API keys)

groq_client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

# ════════════════════════════════════════════════════════════════
#  PART B: LLM-POWERED FINAL REPORT
#  Uses Anthropic API (claude-sonnet-4-6)
#  No API key needed — handled by the environment
# ════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = """You are a personal financial advisor specialising in Indian credit cards.
Your task is to produce a clear, warm, and highly personalised credit card recommendation report.
You will receive a JSON object containing:
  - The user's profile and spending patterns
  - Pre-computed card valuations (do NOT recompute; use the numbers provided)
  - A spend routing table (which card to use for each category)
  - Key benefits and terms for each recommended card

Format your report with these sections (use markdown):
1. **👋 Personal Summary** — greet the user by name, acknowledge their spend profile in 2–3 sentences.
2. **🏆 Top Card Recommendations** — for each recommended card:
   - Card name, issuer, annual fee (note if waived)
   - Exact net annual savings figure (from the data)
   - 3–4 bullet points on WHY this card suits this user specifically
   - A concise "Best used for:" line
3. **💳 Your Spending Wallet** — table showing which card to use for each spend category and why
4. **💡 Pro Tips** — 3–4 tips tailored to this user's lifestyle (e.g. Airtel app, stacking cards)
5. **📋 Terms & Conditions Digest** — for each recommended card list key conditions, caps, and exclusions concisely
6. **✅ Quick Decision** — one-sentence bottom line on which card to get first and why

Tone: conversational, confident, numbers-forward. Use ₹ sign for all amounts. Keep it under 700 words."""

USER_MESSAGE = f"""Here is the user's data. Generate the credit card recommendation report:

{llm_context}"""

print("⏳ Calling Claude API for final report...\n")


try:

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # replace with your model
        temperature=0.2,
        max_tokens=1500,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": USER_MESSAGE
            }
        ]
    )

    report = response.choices[0].message.content

    print("═" * 65)
    print(report)
    print("═" * 65)

except Exception as e:

    print(f"❌ API call failed: {e}")

print("\n📝 Falling back to algorithmic report (Part A).")

print_algorithmic_recommendation(
    card_scores,
    routing,
    spend_analysis,
    USER_PROFILE
)

python-dotenv could not parse statement starting at line 6


⏳ Calling Claude API for final report...

═════════════════════════════════════════════════════════════════
### 👋 Personal Summary
Hi Rahul, I've taken a close look at your spending habits and profile. You have a monthly spend of ₹38,000 across various categories, with a significant portion going towards online shopping, groceries, and dining. Your total annual spend is ₹4,56,000. Given your salaried employment and occasional travel frequency, I've analyzed your spend patterns to provide personalized credit card recommendations.

### 🏆 Top Card Recommendations
Here are my top recommendations for you:
1. **Axis Bank Aura Credit Card** by Axis Bank, with an annual fee of ₹749.
   - Net annual savings: ₹6,091.
   - Why it suits you:
     * Offers rewards and cashback on online shopping, groceries, and dining.
     * Provides partner benefits, including vouchers from Decathlon, Amazon, and IndushealthPlus.
     * Suitable for frequent travelers and health-conscious individuals.
     * Offe